In [ ]:
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from collections import defaultdict
from pathlib import Path
from natsort import natsort_keygen, natsorted # to naturally sort string columns

plt.style.use('matplotlibrc')

KB = 1024
MB = 1024 * 1024
GB = 1024 * 1024 * 1024

In [ ]:
def set_size(fraction_width=0.95, fraction_height=0.25):
    width_pt = 241.14749 # column width in pt
    height_pt = 626.0 # page height in pt

    fig_width_pt = width_pt * fraction_width
    fig_height_pt = height_pt * fraction_height
    inches_per_pt = 1 / 72.27

    fig_width_in = fig_width_pt * inches_per_pt
    fig_height_in = fig_height_pt * inches_per_pt

    return (fig_width_in, fig_height_in)

In [ ]:
# define color to use throughout the notebook
paired = matplotlib.colormaps['Paired']
tab10 = matplotlib.colormaps['tab20']
dark2 = matplotlib.colormaps['Dark2']

c_st =  tab10(0)
c_decomp = tab10(1)
c_rdb_w_pj = tab10(2)
c_rdb_wo_pj = tab10(4)
c_redundancy = tab10(14)

c_rm1 = dark2(0)
c_rm2 = dark2(2)
c_rm3 = dark2(5)
c_rm4 = dark2(3)

colormap = {
    'Single Table': c_st,
    'Decompose': c_decomp,
    'RDB w/ post-join info': c_rdb_w_pj,
    'Result DB': c_rdb_w_pj,
    'RDB w/o post-join info': c_rdb_wo_pj,
    '0. Single Table': c_st,
    'RM1. Dynamic SELECT DISTINCT': c_rm1,
    'RM2. Materialized SELECT DISTINCT': c_rm2,
    'RM3. Dynamic Subquery': c_rm3,
    'RM4. Materialized Subquery': c_rm4,
    'Redundancy': c_redundancy
}

# Result Set Sizes

## Join-Order Benchmark

In [ ]:
def compression_ratio(uncompressed, compressed):
    """
    Compute the compression ratio.

    Returns:
    float: Compression ratio.
    """
    # assert uncompressed != 0, "uncompressed must not be zero"
    if compressed == 0:
        return 0
    return round(uncompressed / compressed, 2)

data = pd.read_csv('../result-set-sizes/job/result-set-sizes.csv')
data = data.drop(['relation', 'count'], axis=1)
data = data.groupby(by=['database', 'query', 'method'], as_index=False).sum()
data = data.sort_values(by=['query', 'method'], key=natsort_keygen())
data['size'] = round(data['size'] / KB, 2)

single_table = data[data['method'] == 'Single Table']
rdb_w_post_join_info = data[data['method'] == 'rdb_w_post_join_info']
rdb_w_post_join_info['compression_ratio'] = [ compression_ratio(baseline, new) for baseline, new in zip(single_table['size'], rdb_w_post_join_info['size']) ]
rdb_wo_post_join_info = data[data['method'] == 'rdb_wo_post_join_info']
rdb_wo_post_join_info['compression_ratio'] = [ compression_ratio(baseline, new) for baseline, new in zip(single_table['size'], rdb_wo_post_join_info['size']) ]

display(single_table)
display(rdb_w_post_join_info)
display(rdb_wo_post_join_info)

## Synthetic Star Schema

In [ ]:
num_dim_tables = 4 # number of dimension tables
dim_payload = 20 # byte
dim_row_size = 4 + dim_payload # 4 byte primary key + payload
dim_table_size = 60
selectivities = np.arange(0.1, 1.1, 0.1)
dim_table_sizes = [ int(dim_table_size * sel) for sel in selectivities ]

fact_payload = 20
fact_row_size = 4 + num_dim_tables * 4 + fact_payload # 4 byte pk + 4 byte foreign key per dimension table + payload
fact_size = dim_table_size**num_dim_tables
fact_table_sizes = [ dim_size**num_dim_tables for dim_size in dim_table_sizes ]

single_table = [ ((fact_row_size + dim_row_size * num_dim_tables) * f_size) / MB for f_size in fact_table_sizes ]
rdb_with_post_join = ([ ((f_size * fact_row_size) + (dim_row_size * d_size * num_dim_tables)) / MB
                       for f_size, d_size in zip(fact_table_sizes, dim_table_sizes) ])
rdb_without_post_join = [ (dim_payload * d_size * num_dim_tables + f_size * (fact_payload)) / MB for f_size, d_size in zip(fact_table_sizes, dim_table_sizes) ]

### Plot
fig, ax = plt.subplots(figsize=set_size(fraction_width=1.0, fraction_height=0.22), layout='constrained')
ax.plot(selectivities, single_table, 'o--', color=colormap['Single Table'], label='Single Table')
ax.plot(selectivities, rdb_with_post_join, 'x--', color=colormap['RDB w/ post-join info'], label='RDB w/ post-join info')
ax.plot(selectivities, rdb_without_post_join, '^--', color=colormap['RDB w/o post-join info'], label='RDB w/o post-join info')

ax.fill_between(selectivities, single_table, rdb_with_post_join, color=colormap['Redundancy'], label='Redundancy')

ax.set_xlabel('Selectivity')
ax.set_ylabel('Result Set Size [MiB]')
ax.legend(loc='upper left', ncols=1)

fig.savefig('synthetic-result-sizes.pdf', bbox_inches='tight')

# Rewrite Methods

## Join-Order Benchmark

In [ ]:
def preprocess_data(data):
    # construct the following dictionary:
    # {
    #   'RM0': [q0_time, q1_time, ..., qn-1_time]
    #   'RM1': [q0_time, q1_time, ..., qn-1_time]
    # }
    data = data.sort_values(by=['query', 'method'], key=natsort_keygen()) # ensure that data is sorted correctly
    method_times = defaultdict(list)
    for _, row in data.iterrows():
        method_times[row['method']].append(row['time'])
    return method_times

def bar_plot(data, data_transfer, filename):
    queries = natsorted(data['query'].unique())
    method_times = preprocess_data(data[data['data_transfer'] == data_transfer])
    x = np.arange(len(queries))  # the label locations
    width = 0.20  # the width of the bars
    multiplier = 0
    fig, ax = plt.subplots(figsize=set_size(fraction_width=1, fraction_height=0.26), layout='constrained')
    for method, times in method_times.items():
        offset = width * multiplier
        rects = ax.bar(x + offset, times, width, color=colormap[method], label=method)
        multiplier += 1
    ax.set_xticks(x + width + width/2, queries)
    ax.set_xlabel('JOB Queries')
    ax.legend(loc='upper left')
    ax.xaxis.grid(False) # Disable grid lines on the x-axis
    
    log_scale = True
    if log_scale:
        ax.set_ylabel('Query Execution Time [ms] (log-scale)')
        ax.set_yscale('log')
    else:
        ax.set_ylabel('Query Execution Time [ms]')
        ax.set_ylim(0)
        
    fig.savefig(filename, bbox_inches='tight')

data = pd.read_csv('../rewrite-methods/job/rewrite-results.csv')
data = data.groupby(by=['database', 'system', 'query', 'method', 'data_transfer', 'run'], as_index=False).sum()
data = data.groupby(by=['database', 'system', 'query', 'method', 'data_transfer'], as_index=False).median().drop(['run', 'num_query_internal'], axis=1)
data = data.drop(['database', 'system'], axis=1)
exclude_queries = [
    # 'q13a',
    # 'q17a'
]
data = data[~data['query'].isin(exclude_queries)]
exclude_methods = [
    '0. Single Table'
]
data = data[~data['method'].isin(exclude_methods)]

bar_plot(data, False, 'job-rewrite-methods.pdf')

In [ ]:
def overhead(baseline, new):
    assert baseline != 0, "baseline must not be zero"
    if (new <= baseline): # new is faster -> improvement
        return -((baseline - new) / baseline) * 100
    else: # new is slower -> overhead 
        return ((new - baseline) / baseline) * 100
        
data = pd.read_csv('../rewrite-methods/job/rewrite-results.csv')
data = data.groupby(by=['database', 'system', 'query', 'method', 'data_transfer', 'run'], as_index=False).sum()
data = data.groupby(by=['database', 'system', 'query', 'method', 'data_transfer'], as_index=False).median().drop(['run', 'num_query_internal'], axis=1)
data = data.drop(['database', 'system'], axis=1)
data = data[data['data_transfer'] == False]
exclude_queries = [
    # 'q13a',
]
data = data[~data['query'].isin(exclude_queries)]
# methods = data['method'].unique()[1:]

single_table = data[data['method'] == '0. Single Table']
rewrite_methods = data[data['method'] != '0. Single Table']

# NOTE: be careful as ties are included (handle them manually)
min_values = rewrite_methods.groupby(by=['query'])['time'].transform('min')
best_rewrite_method = rewrite_methods[rewrite_methods['time'] == min_values]

# ensure that both dataframes are sorted correctly
single_table = single_table.sort_values(by=['query', 'method'], key=natsort_keygen()) # ensure that data is sorted correctly
best_rewrite_method = best_rewrite_method.sort_values(by=['query', 'method'], key=natsort_keygen()) # ensure that data is sorted correctly

assert len(single_table) == len(best_rewrite_method)

for (index1, st), (index2, rm) in zip(single_table.iterrows(), best_rewrite_method.iterrows()):
    print(f"{st['query']}: {round(overhead(int(st['time']), int(rm['time'])), 1)}")

## Synthetic

### Varying selectivity

In [ ]:
def preprocess_data(data):
    # construct the following dictionary:
    # {
    #   'RM0': [q0_time, q1_time, ..., qn-1_time]
    #   'RM1': [q0_time, q1_time, ..., qn-1_time]
    # }
    data = data.sort_values(by=['query', 'method'], key=natsort_keygen()) # ensure that data is sorted correctly
    method_times = defaultdict(list)
    for _, row in data.iterrows():
        method_times[row['method']].append(row['time'])
    return method_times

def bar_plot(data, data_transfer, filename):
    queries = natsorted(data['query'].unique())
    selectivities = [ int(q.rsplit('_', 1)[1]) / 100 for q in data['query'].unique() ]
    method_times = preprocess_data(data[data['data_transfer'] == data_transfer])
    x = np.arange(len(selectivities))  # the label locations
    width = 0.20  # the width of the bars
    multiplier = 0
    fig, ax = plt.subplots(figsize=set_size(fraction_width=1, fraction_height=0.26), layout='constrained')
    for method, times in method_times.items():
        offset = width * multiplier
        rects = ax.bar(x + offset, times, width, color=colormap[method], label=method)
        multiplier += 1
    selectivities = [ int(q.rsplit('_', 1)[1]) / 100 for q in queries ]
    ax.set_xticks(x + width + width/2, selectivities)
    ax.set_xlabel('Selectivity')
    ax.xaxis.grid(False) # Disable grid lines on the x-axis
    ax.legend()

    log_scale = True
    if log_scale:
        ax.set_ylabel('Query Execution Time [ms] (log-scale)')
        ax.set_yscale('log')
    else:
        ax.set_ylabel('Query Execution Time [ms]')
        ax.set_ylim(0)
    
    fig.savefig(filename, bbox_inches='tight')

data = pd.read_csv('../rewrite-methods/star/rewrite-results.csv')
data = data[data['query'].str.contains('star_sel_')]
data = data.groupby(by=['database', 'system', 'query', 'method', 'data_transfer', 'run'], as_index=False).sum()
data = data.groupby(by=['database', 'system', 'query', 'method', 'data_transfer'], as_index=False).median().drop(['run', 'num_query_internal'], axis=1)
data = data.drop(['database', 'system'], axis=1)

exclude_methods = [
    '0. Single Table'
]
data = data[~data['method'].isin(exclude_methods)]

bar_plot(data, False, 'synthetic-rewrite-methods-selectivity.pdf')

In [ ]:
def overhead(baseline, new):
    assert baseline != 0, "baseline must not be zero"
    if (new <= baseline): # new is faster -> improvement
        return -((baseline - new) / baseline) * 100
    else: # new is slower -> overhead 
        return ((new - baseline) / baseline) * 100
        
data = pd.read_csv('../rewrite-methods/star/rewrite-results.csv')
data = data.groupby(by=['database', 'system', 'query', 'method', 'data_transfer', 'run'], as_index=False).sum()
data = data.groupby(by=['database', 'system', 'query', 'method', 'data_transfer'], as_index=False).median().drop(['run', 'num_query_internal'], axis=1)
data = data.drop(['database', 'system'], axis=1)
data = data[data['query'].str.contains('star_sel_')]
data = data[data['data_transfer'] == False]

single_table = data[data['method'] == '0. Single Table']
rewrite_methods = data[data['method'] != '0. Single Table']

# NOTE: be careful as ties are included (handle them manually)
min_values = rewrite_methods.groupby(by=['query'])['time'].transform('min')
best_rewrite_method = rewrite_methods[rewrite_methods['time'] == min_values]

# ensure that both dataframes are sorted correctly
single_table = single_table.sort_values(by=['query', 'method'], key=natsort_keygen()) # ensure that data is sorted correctly
best_rewrite_method = best_rewrite_method.sort_values(by=['query', 'method'], key=natsort_keygen()) # ensure that data is sorted correctly

assert len(single_table) == len(best_rewrite_method)

for (index1, st), (index2, rm) in zip(single_table.iterrows(), best_rewrite_method.iterrows()):
    print(f"{st['query']}: {round(overhead(int(st['time']), int(rm['time'])), 1)}")

# RESULTDB Algorithm

## Join-Order Benchmark

In [ ]:
# construct the following dictionary:
# {
#   'Single-Table': [q4_time, q5_time, ...]
#   'Single-Table + Decompose': [q4_time, q5_time, ...]
#   'Result-DB': [q4_time, q5_time, ...]
# }
# ensure that `queries` is sorted!
queries = [
    'q1a',
    'q3b',
    'q4a',
    'q5b',
]

single_table_decompose = []
resultdb = []
processed_queries = []
num_queries = 0
for query in queries:
    result_file = Path(f"../algorithm/job/{query}_results.csv")
    if not result_file.is_file():
        print(f"{result_file} does not exist. Skipping...")
        continue
    processed_queries.append(query)
    data = pd.read_csv(result_file)  
    data = (data.groupby(by=['commit', 'date', 'version', 'suite', 'benchmark', 'experiment', 'name', 'config', 'case',], as_index=False)
                .median()
                .drop(['commit', 'date', 'version', 'suite', 'benchmark', 'case', 'runid'], axis=1)
           )
    experiment = data['experiment'].unique()
    assert len(experiment) == 1, f'experiment contains multiple different queries'
    assert experiment[0] == query, f'experiment query: {experiment[0]} does not match query {query} of current file'

    for _, row in data.iterrows():
        algorithm = row['name']
        execution_time = row['time']
        if "single-table" in algorithm:
            single_table_execution_time = execution_time
        elif "decompose" in algorithm:
            decompose_execution_time = execution_time
        elif "resultdb" in algorithm:
            resultdb_execution_time = execution_time
        else:
            assert False, "experiment name: {name} does not match any of our algorithms"
            
    single_table_decompose.append((single_table_execution_time, decompose_execution_time - single_table_execution_time))
    resultdb.append(resultdb_execution_time)

assert len(single_table_decompose) == len(resultdb) == len(processed_queries), f'number of measurements has to match the processed queries'

# plot data
x = np.arange(len(processed_queries))  # the label locations
width = 0.30  # the width of the bars
fig, ax = plt.subplots(figsize=set_size(fraction_width=1, fraction_height=0.20), layout='constrained')

st_time = [ st_decomp[0] for st_decomp in single_table_decompose ]
decomp_time = [ st_decomp[1] for st_decomp in single_table_decompose ]
ax.bar(x, st_time, width, color=colormap['Single Table'], edgecolor='black', label='Single Table')
ax.bar(x, decomp_time, width, bottom=st_time, color=colormap['Decompose'], edgecolor='black', label='Decompose')

offset = width
ax.bar(x + offset, resultdb, width, color=colormap['Result DB'], edgecolor='black', label='Result DB')

ax.set_ylabel('Query Execution Time [ms]')
# ax.set_title('JOB Algorithm Runtime in mutable')
ax.set_xlabel('JOB Queries')
ax.set_xticks(x + width / 2, processed_queries)
ax.legend(loc='upper right')
ax.set_ylim(0)
ax.xaxis.grid(False) # Disable grid lines on the x-axis

fig.savefig('job-algorithm.pdf', bbox_inches='tight')

## Synthetic

### Varying selectivity

In [ ]:
# ensure that `queries` is sorted!
queries = [
    'star_sel_10',
    'star_sel_20',
    'star_sel_30',
    'star_sel_40',
    'star_sel_50',
    'star_sel_60',
    'star_sel_70',
    'star_sel_80',
    'star_sel_90',
    'star_sel_100',
]
single_table_decompose = []
resultdb = []
processed_queries = []
num_queries = 0

for query in queries:
    result_file = Path(f"../algorithm/star/{query}_results.csv")
    if not result_file.is_file():
        print(f"{result_file} does not exist. Skipping...")
        continue
    processed_queries.append(query)
    data = pd.read_csv(result_file)  
    data = (data.groupby(by=['commit', 'date', 'version', 'suite', 'benchmark', 'experiment', 'name', 'config', 'case',], as_index=False)
                .median()
                .drop(['commit', 'date', 'version', 'suite', 'benchmark', 'case', 'runid'], axis=1)
           )
    experiment = data['experiment'].unique()
    assert len(experiment) == 1, f'experiment contains multiple different queries'
    assert experiment[0] == query or experiment[0] in query, f'experiment query: {experiment[0]} does not match query {query} of current file'
    
    for _, row in data.iterrows():
        algorithm = row['name']
        execution_time = row['time']
        if "single-table" in algorithm:
            single_table_execution_time = execution_time
        elif "decompose" in algorithm:
            decompose_execution_time = execution_time
        elif "resultdb" in algorithm:
            resultdb_execution_time = execution_time
        else:
            assert False, "experiment name: {name} does not match any of our algorithms"
            
    single_table_decompose.append((single_table_execution_time, decompose_execution_time - single_table_execution_time))
    resultdb.append(resultdb_execution_time)

assert len(single_table_decompose) == len(resultdb) == len(processed_queries), f'number of measurements has to match the processed queries'

# plot data
x = np.arange(len(processed_queries))  # the label locations
width = 0.30  # the width of the bars
multiplier = 0
fig, ax = plt.subplots(figsize=set_size(fraction_width=1, fraction_height=0.20), layout='constrained')

st_time = [ st_decomp[0] for st_decomp in single_table_decompose ]
decomp_time = [ st_decomp[1] for st_decomp in single_table_decompose ]
ax.bar(x, st_time, width, color=colormap['Single Table'], edgecolor='black', label='Single Table')
ax.bar(x, decomp_time, width, bottom=st_time, color=colormap['Decompose'], edgecolor='black', label='Decompose')

offset = width
ax.bar(x + offset, resultdb, width, color=colormap['Result DB'], edgecolor='black', label='Result DB')

ax.set_ylabel('Query Execution Time [ms]')
# ax.set_title('Synthetic Algorithm Runtime in mutable')
selectivities = [ int(q.rsplit('_', 1)[1]) / 100 for q in processed_queries ]
ax.set_xticks(x + width / 2, selectivities)
ax.set_xlabel('Selectivity')
ax.legend(loc='upper left')
ax.set_ylim(0)
ax.xaxis.grid(False) # Disable grid lines on the x-axis

fig.savefig('synthetic-algorithm.pdf', bbox_inches='tight')